## Deep Research

Jeden z klasycznych, uniwersalnych biznesowo przypadków użycia Agentic AI! To jest naprawdę duże.

**Ta wersja jest inna niż oryginał: zamiast frameworka OpenAI Agents SDK, budujemy dokładnie te same mechanizmy orkiestracji ręcznie, na natywnym Anthropic SDK.** Tak jak w poprzednich notatnikach tego tygodnia, żadna z abstrakcji tego labu (`Agent`, `Runner`, `WebSearchTool`, `function_tool`, `output_type`) nie ma odpowiednika w Anthropic SDK. Redefiniujemy lokalnie `trace()`, `run()` i `handle_tool_calls()` analogicznie do `2_lab2.pl.ipynb` i `3_lab3.pl.ipynb` (każdy notatnik tego kursu jest samodzielnym projektem), tym razem rozszerzone o dwa nowe mechanizmy: server-side narzędzie `web_search` (hostowane po stronie Anthropic, odpowiednik `WebSearchTool` z OpenAI) i `anthropic.messages.parse(output_format=...)` jako alternatywną ścieżkę w TYM SAMYM `run()`, nie osobną funkcję jak w lab3 - bo w tym labie różne "agenty" naprzemiennie potrzebują zwykłego tekstu, structured output i narzędzi.

**Inna różnica: `run()` zwraca tu tylko wynik (tekst albo sparsowany obiekt), nie krotkę `(tekst, historia)` jak w `2_lab2.pl.ipynb`/`3_lab3.pl.ipynb`.** Żadne dwa wywołania w tym notatniku nie dzielą ze sobą historii rozmowy (każdy z czterech "agentów" jest wywoływany niezależnie, dokładnie jak w oryginale przez osobne `Runner.run()`), więc historia nie jest tu nigdzie potrzebna.

**Świadoma decyzja: literalne przykładowe zapytania (`task`, `query`) zostają po angielsku.** Reszta notatnika (instrukcje, szablony promptów, komunikaty) jest przetłumaczona na polski zgodnie z konwencją repo - ale to konkretne zapytanie trafia do prawdziwej wyszukiwarki internetowej przez narzędzie `web_search`, a temat ("Most popular AI Agent frameworks in 2026") jest zdominowany przez anglojęzyczne źródła. Polskie zapytanie wyszukiwania realnie pogorszyłoby jakość wyników - to samo dotyczy `INSTRUCTIONS` Planner Agenta: mimo że są po polsku, Claude może w ich efekcie generować zapytania wyszukiwania po polsku, co obniży trafność wyników dla tego konkretnego tematu. Jeśli zauważysz słabe wyniki wyszukiwania, rozważ dodanie do instrukcji Plannera zdania w stylu "formułuj zapytania wyszukiwania po angielsku".

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Implikacje komercyjne</h2>
            <span style="color:#00bfff;">Agent Deep Research ma szerokie zastosowanie w praktycznie każdym obszarze biznesowym, a także w Twoich codziennych zadaniach. Możesz go wykorzystać sam/sama!
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Importy tej wersji notatnika. Zamiast biblioteki agents (OpenAI Agents SDK) importujemy wyłącznie natywny klient Anthropic, w wariancie asynchronicznym.
# AsyncAnthropic jest potrzebny, bo run_searches() niżej odpala wiele wyszukiwań naraz przez asyncio.gather - dokładnie ten sam powód co w 2_lab2.pl.ipynb.
# pydantic zostaje bez zmian - BaseModel i Field są niezależne od dostawcy LLM; te same klasy posłużą do structured output przez anthropic.messages.parse().
# contextmanager i time odtwarzają trace() z poprzednich notatników, a json posłuży do serializacji wyników narzędzia send_email_tool zwracanych do Claude (redefiniowane tutaj, bo każdy notatnik tego kursu jest samodzielnym projektem).
# display i Markdown renderują wynik Search Agenta jako sformatowany markdown w komórce notatnika, niezależnie od dostawcy LLM.
# messenger.py to gotowy moduł wysyłki (e-mail/push), w 100% niezależny od dostawcy LLM - ten sam moduł co w 2_lab2.pl.ipynb/3_lab3.pl.ipynb.

from anthropic import AsyncAnthropic  # asynchroniczny klient Anthropic - zastępuje framework agents
from pydantic import BaseModel, Field  # definicje structured output (WebSearchPlan, ReportData) - niezależne od dostawcy
from dotenv import load_dotenv  # wczytuje zmienne środowiskowe (klucze API) z pliku .env
import asyncio  # asyncio.gather do równoległego wykonania wyszukiwań w run_searches()
import json  # serializacja wyników narzędzia send_email_tool do formatu JSON
import time  # pomiar czasu wewnątrz ręcznego trace()
from contextlib import contextmanager  # dekorator do napisania trace() jako context managera
from IPython.display import display, Markdown  # renderowanie wyniku Search Agenta jako markdown w notatniku
from messenger import send_email, push  # gotowe funkcje wysyłki e-mail/push, niezależne od dostawcy LLM

In [ ]:
# Wczytanie kluczy z .env i inicjalizacja klienta Anthropic - klucz brany automatycznie z ANTHROPIC_API_KEY.
# Klient jest asynchroniczny (AsyncAnthropic), bo run() niżej używa await, a run_searches() woła kilka wyszukiwań naraz przez asyncio.gather.
# W oryginale ten krok odpowiada niejawnemu utworzeniu klienta OpenAI wewnątrz każdego obiektu Agent - tutaj tworzymy jeden, współdzielony klient raz, na początku notatnika.
# load_dotenv(override=True) musi wykonać się PRZED użyciem AsyncAnthropic(), żeby zmienna ANTHROPIC_API_KEY była już w środowisku.

load_dotenv(override=True)  # ładuje .env i nadpisuje istniejące zmienne środowiskowe
anthropic = AsyncAnthropic()  # klient Anthropic - odpowiednik utworzenia agentów z frameworka agents

In [ ]:
# Stałe konfiguracyjne notatnika - MODEL_NAME z oryginału zastąpiony przez MODEL z jednym, tanim modelem Claude.
# W oryginale MODEL_NAME to "gpt-5.4-mini" - tutaj, zgodnie z pułapem kosztowym tego repo, używamy najtańszego dostępnego modelu Claude przez cały notatnik.
# USE_EMAIL i HOW_MANY_SEARCHES zostają bez zmian - są niezależne od dostawcy LLM.
# Wszystkie cztery "agenty" w tym notatniku (Search, Planner, Writer, Email) współdzielą tę samą stałą MODEL, tak jak w oryginale każdy Agent(...) dostawał ten sam MODEL_NAME.

MODEL = "claude-haiku-4-5"  # najtańszy dostępny model - pułap kosztowy na czas przechodzenia przez kurs
USE_EMAIL = True  # wymuś próbę wysyłki e-mailem (bez automatycznego fallbacku na Pushover)
HOW_MANY_SEARCHES = 5  # ile zapytań wyszukiwania ma wygenerować Planner Agent

## Strategia dla Agenta Deep Research

Zrobimy to w sposób odporny na błędy (bulletproof).

Zorkiestrujemy to przez kod: osobne wywołania `run()` dla każdego kroku procesu (odpowiednik `Runner.run()` z oryginału).

W każdym punkcie użyjemy Structured Outputs.

## Zbudujemy 4 Agentów:

1. Search Agent: przeszukuje sieć w poszukiwaniu informacji
2. Planner Agent: na podstawie pytania układa listę wyszukiwań do wykonania
3. Writer Agent: pisze solidny raport
4. Emailer Agent: układa i wysyła e-mail

A potem 4 funkcje Pythona, po jednej do wywołania `run()` dla każdego z 4 agentów.

## Agent 1: Search Agent

### Hostowane narzędzia OpenAI (kontekst z oryginału)

https://openai.github.io/openai-agents-python/tools/#hosted-tools

Płatne, szybkie podejście do wykonywania zarządzanej funkcjonalności w chmurze OpenAI.

Ich dokumentacja pokazuje te narzędzia, ale warto pamiętać, że są kosztowne i zamykają Cię w ekosystemie OpenAI.

OpenAI oferuje następujące hostowane narzędzia:

`WebSearchTool` pozwala agentowi przeszukiwać sieć.
`FileSearchTool` pozwala pobierać informacje z Twoich OpenAI Vector Stores.
`CodeInterpreterTool` pozwala LLM-owi wykonywać kod w środowisku sandboksowym.
`HostedMCPTool` udostępnia modelowi narzędzia zdalnego serwera MCP.
`ImageGenerationTool` generuje obrazy na podstawie promptu.
`ToolSearchTool` pozwala modelowi doładowywać odroczone narzędzia, namespace'y albo hostowane serwery MCP na żądanie.

### Ważna uwaga - koszt WebSearchTool w API OpenAI

To obecnie kosztuje 1 centa za wywołanie dla WebSearchTool OpenAI. To może się złożyć na około $1 przy kolejnych 2 labach. W kolejnych labach użyjemy darmowych i tanich narzędzi wyszukiwania na innych platformach, więc możesz spokojnie pominąć uruchomienie tego, jeśli koszt Cię niepokoi. Student Christian W. zwrócił też uwagę, że OpenAI czasem nalicza opłatę za wiele wyszukiwań w ramach jednego wywołania, więc czasem może to kosztować więcej niż 1 cent za wywołanie.

Koszty są w sekcji Tools tutaj: https://developers.openai.com/api/docs/pricing

**Odpowiednik Anthropic: narzędzie `web_search`, też server-side, ale z osobnym cennikiem.** Anthropic hostuje wyszukiwanie po swojej stronie analogicznie do `WebSearchTool` - deklarujesz je w `tools=`, a Claude sam decyduje (albo jest do tego wymuszony przez `tool_choice`), kiedy go użyć; wynik trafia z powrotem jako blok `web_search_tool_result` w tej samej turze, bez pętli po Twojej stronie. Cennik Anthropic za wyszukiwanie jest inny niż powyższy 1 cent OpenAI - sprawdź aktualny cennik w dokumentacji Anthropic, jeśli koszt ma dla Ciebie znaczenie. Używamy tu wariantu `web_search_20250305` (podstawowa wersja, bez dynamic filtering) - Claude Haiku 4.5 (domyślny model tego repo) nie wspiera nowszego wariantu `web_search_20260209`, dostępnego tylko dla modeli Opus/Sonnet z rodziny 4.6+.

In [ ]:
# Instrukcje Search Agenta - przetłumaczone, bo to prompt systemowy czytany przez model.
# task to przykładowe zapytanie testowe używane w demo niżej (i ponownie w demo Planner Agenta) - CELOWO zostaje po angielsku, patrz notatka architektoniczna w komórce 0.
# tool_choice zastępuje ModelSettings(tool_choice="required") z oryginału - {"type": "any"} wymusza użycie JAKIEGOŚ narzędzia (tu jedynego: web_search).
# search_tools to lista schematów narzędzi Anthropic - web_search_20250305 to server-side narzędzie hostowane przez Anthropic, odpowiednik WebSearchTool() z oryginału.
# Wariant 20250305 (bez dynamic filteringu) jest celowy - patrz uwaga w komórce markdown wyżej o wsparciu modeli dla wariantu 20260209.

INSTRUCTIONS = """
Jesteś asystentem badawczym. Otrzymując termin wyszukiwania, przeszukujesz sieć w poszukiwaniu tego terminu i
tworzysz zwięzłe podsumowanie wyników. Podsumowanie musi mieć 2-3 akapity i mniej niż 300 słów.
Uchwyć główne punkty i bądź zwięzły/a. Odpowiedz tylko podsumowaniem.
"""  # prompt systemowy Search Agenta
task = "Most popular AI Agent frameworks in 2026"  # przykładowe zapytanie testowe - celowo po angielsku (patrz notatka architektoniczna w komórce 0)

search_tool_choice = {"type": "any"}  # wymuś użycie narzędzia - odpowiednik ModelSettings(tool_choice="required")
search_tools = [{"type": "web_search_20250305", "name": "web_search"}]  # server-side narzędzie wyszukiwania Anthropic - odpowiednik WebSearchTool()

In [ ]:
# W oryginale Agent(name=..., instructions=..., tools=tools, model=..., model_settings=settings) tworzy obiekt agenta z frameworka OpenAI Agents SDK.
# W naszej wersji "agent" to po prostu string instrukcji (system prompt) - dokładnie ten sam wzorzec co w 2_lab2.pl.ipynb/3_lab3.pl.ipynb.
# tools i tool_choice zostały zdefiniowane osobno w komórce wyżej (search_tools, search_tool_choice) i będą przekazywane do run() przy każdym wywołaniu tego agenta.
# Ta zmienna nie wykonuje żadnego wywołania API - to tylko przypisanie stringa, dokładnie jak sales_agent1/2/3 we wcześniejszych notatnikach.

search_agent = INSTRUCTIONS  # "Search Agent" - w naszej wersji to sam string instrukcji

### Mechanizmy z poprzednich notatników, rozszerzone o narzędzia server-side i structured output

Redefiniujemy `trace()`, `run()` i `handle_tool_calls()` analogicznie do `2_lab2.pl.ipynb`/`3_lab3.pl.ipynb` (każdy notatnik tego kursu jest samodzielnym projektem). Nowość w tym notatniku: `run()` dostaje parametr `output_format` - gdy jest podany, wywołanie leci przez `anthropic.messages.parse()` zamiast zwykłego `messages.create()`, a funkcja zwraca od razu sparsowany obiekt Pydantic (Planner i Writer Agent). Gdy `output_format` nie jest podany, `run()` działa jak zwykła pętla tool-use (Search Agent - narzędzie server-side, Email Agent - narzędzie kliencie) i zwraca finalny tekst. `handle_tool_calls()` jest tu uproszczone względem `2_lab2.pl.ipynb`: bez rozróżnienia handoff/narzędzie (ten lab nie ma agent-jako-narzędzie), za to bez `iscoroutinefunction` też - jedyne narzędzie kliencie w tym notatniku (`send_email_tool`) jest synchroniczne.

Znane uproszczenie: `run()` nie obsługuje jawnie `stop_reason == "pause_turn"` (długie łańcuchy wyszukiwań web_search mogą teoretycznie przerwać turę do wznowienia) - dla pojedynczego wymuszonego wyszukiwania w tym labie to nie występuje w praktyce, ale przy rozbudowie na wiele wyszukiwań w jednej turze rozważ dopisanie obsługi analogicznej do wzorca z dokumentacji Anthropic (manual agentic loop).

In [ ]:
# Ta komórka definiuje trace() - dokładnie ten sam, lekki, lokalny odpowiednik obserwowalności co w poprzednich notatnikach.
# @contextmanager pozwala napisać funkcję generatorową, którą Python zamienia w obiekt obsługujący "with trace(...): ...".
# Kod przed yield wykonuje się przy wejściu do bloku with, kod po yield - przy wyjściu z niego, nawet jeśli w środku wystąpi wyjątek.
# Redefiniujemy ją tutaj, a nie importujemy z innego notatnika, bo każdy notatnik tego kursu jest samodzielnym, niezależnym projektem.

@contextmanager
def trace(name: str):  # lokalny, uproszczony odpowiednik trace() z OpenAI Agents SDK - bez wysyłki danych na zewnątrz
    start = time.time()  # zapamiętaj moment startu, żeby policzyć czas trwania
    print(f"[trace] start: {name}")  # znacznik początku
    yield  # tutaj wykonuje się kod wewnątrz bloku "with trace(...):"
    print(f"[trace] koniec: {name} ({time.time() - start:.2f}s)")  # znacznik końca razem z czasem trwania

In [ ]:
# handle_tool_calls() wykonuje wywołania narzędzi kliencie (tu tylko send_email_tool) i buduje listę bloków tool_result.
# Uproszczone względem 2_lab2.pl.ipynb: bez flagi is_handoff (ten lab nie ma agent-jako-narzędzie) i bez iscoroutinefunction (jedyne narzędzie kliencie tutaj jest synchroniczne).
# Narzędzia server-side (web_search) NIGDY tu nie trafiają - Claude wykonuje je sam po swojej stronie, więc stop_reason nie ustawia się na "tool_use" dla web_search.
# run() to async odpowiednik Agent + Runner.run(), rozszerzony o parametr output_format - gałąź structured output leci przez anthropic.messages.parse() i zwraca sparsowany obiekt Pydantic, bez wchodzenia w pętlę tool-use.
# Gdy output_format nie jest podany, run() działa jak zwykła pętla tool-use (jak w poprzednich notatnikach) i zwraca finalny tekst - ten jeden helper obsłuży wszystkie cztery agenty tego notatnika.

async def handle_tool_calls(tool_use_blocks: list) -> list[dict]:  # wykonuje wywołania narzędzi kliencie i buduje tool_result
    results = []  # lista bloków tool_result do wysłania z powrotem
    for block in tool_use_blocks:  # iteruj po każdym bloku tool_use z odpowiedzi
        tool = globals().get(block.name)  # znajdź funkcję Pythona o tej samej nazwie co narzędzie
        output = tool(**block.input) if tool else f"Nieznane narzędzie: {block.name}"  # wykonaj narzędzie albo zwróć komunikat błędu
        results.append({
            "type": "tool_result",  # Anthropic: blok tool_result zamiast wiadomości z rolą "tool" jak w OpenAI
            "tool_use_id": block.id,  # musi się zgadzać z id bloku tool_use, na który odpowiadamy
            "content": json.dumps(output),  # treść wyniku jako string JSON
        })
    return results  # zwracane bloki trafią razem do JEDNEJ wiadomości user


async def run(instructions: str, user_message: str, tools: list | None = None, tool_choice: dict | None = None, output_format: type[BaseModel] | None = None):  # async odpowiednik Agent + Runner.run(), rozszerzony o output_format
    messages = [{"role": "user", "content": user_message}]  # pojedyncza wiadomość - ten notatnik nie wątkuje historii między wywołaniami (patrz notatka w komórce 0)
    kwargs = {"model": MODEL, "max_tokens": 16000, "system": instructions, "messages": messages}  # wspólne argumenty dla obu ścieżek (structured output i zwykła pętla)
    if tools:  # niektórzy agenci (Planner, Writer) nie dostają żadnych narzędzi
        kwargs["tools"] = tools  # dorzuć listę narzędzi tylko, gdy faktycznie podana
    if tool_choice:  # wymuszenie użycia narzędzia bywa None (domyślne zachowanie Claude)
        kwargs["tool_choice"] = tool_choice  # np. {"type": "any"} dla Search Agenta
    if output_format:  # Planner Agent i Writer Agent - odpowiednik output_type= z Agents SDK
        response = await anthropic.messages.parse(output_format=output_format, **kwargs)  # generuje schemat JSON z klasy Pydantic i waliduje odpowiedź względem niego
        return response.parsed_output  # gotowy, zwalidowany obiekt Pydantic (odpowiednik result.final_output przy output_type=...)
    response = await anthropic.messages.create(**kwargs)  # zwykłe wywołanie - Search Agent (narzędzie server-side) albo Email Agent (narzędzie kliencie)
    while response.stop_reason == "tool_use":  # pętla trwa, dopóki Claude chce użyć narzędzia KLIENCIEGO (web_search server-side nigdy nie trafia w tę pętlę)
        tool_use_blocks = [block for block in response.content if block.type == "tool_use"]  # wyciągnij bloki tool_use z odpowiedzi
        results = await handle_tool_calls(tool_use_blocks)  # wykonaj narzędzia kliencie, zbuduj tool_result
        messages.append({"role": "assistant", "content": response.content})  # cała odpowiedź assistant wraca do historii
        messages.append({"role": "user", "content": results})  # wszystkie wyniki narzędzi w JEDNEJ wiadomości user
        response = await anthropic.messages.create(**kwargs)  # kwargs["messages"] to ten sam obiekt messages, więc widzi dopisane wyżej wpisy
    return next(block.text for block in response.content if block.type == "text")  # finalna odpowiedź tekstowa (content[0] bywa ThinkingBlock albo web_search_tool_result)

In [ ]:
# Uruchomienie Search Agenta - Claude wymusza użycie narzędzia web_search (server-side), a wynik trafia od razu jako tekst finalny (bez pętli tool_use po naszej stronie).
# display(Markdown(...)) renderuje wynik jako sformatowany markdown w komórce - niezależne od dostawcy LLM.
# Odpowiednik oryginalnego result = await Runner.run(search_agent, task); display(Markdown(result.final_output)) - tutaj run() zwraca od razu gotowy tekst zamiast obiektu result z atrybutem final_output.
# To wywołanie kosztuje realne zapytanie do wyszukiwarki po stronie Anthropic - patrz uwaga o cenniku w komórce markdown wyżej.

result = await run(search_agent, task, tools=search_tools, tool_choice=search_tool_choice)  # wymuszone wyszukiwanie w sieci
display(Markdown(result))  # wyświetl wynik jako markdown

### Nie ma tu prawdziwego "trace" do obejrzenia

W oryginale KAŻDE wywołanie `Runner.run()` jest automatycznie śledzone przez platformę OpenAI, niezależnie od jawnego bloku `with trace(...):` - stąd link do `platform.openai.com/traces` pojawia się w oryginale nawet po wywołaniach spoza takiego bloku, jak demo wyżej. W naszej wersji `trace()` to tylko lokalny context manager (zdefiniowany wyżej) - działa wyłącznie tam, gdzie jawnie opakujesz wywołanie w `with trace(...):`, i wypisuje tylko dwie linie w konsoli (start/koniec + czas trwania). Zobaczysz go w akcji w sekcji Showtime na końcu notatnika - demo wyżej nie jest nim opakowane, tak jak w oryginale.

## Agent 2: Planner Agent

### Teraz użyjemy Structured Outputs, razem z opisem pól

In [ ]:
# Definicje obiektów Pythona (schematów), które ma wypełnić Claude - klasy Pydantic, w 100% niezależne od dostawcy LLM, zostają bez zmian względem oryginału.
# Field(description=...) to tekst czytany przez model przy wypełnianiu schematu, więc tłumaczymy go na polski, tak samo jak prompty i opisy narzędzi.
# WebSearchPlan.searches to lista WebSearchItem - dokładnie ta struktura trafi do output_format= w run() niżej.
# Nazwy klas i pól (WebSearchItem, reason, query, searches) zostają po angielsku - to składnia kodu, nie tekst czytany przez człowieka wprost, więc nie podlega tłumaczeniu.

class WebSearchItem(BaseModel):  # pojedyncze zaplanowane wyszukiwanie
    reason: str = Field(description="Twoje uzasadnienie, dlaczego to wyszukiwanie jest ważne dla zapytania.")  # pole tekstowe - uzasadnienie
    query: str = Field(description="Termin wyszukiwania do użycia w wyszukiwarce internetowej.")  # pole tekstowe - sam termin


class WebSearchPlan(BaseModel):  # pełny plan wyszukiwań zwracany przez Planner Agenta
    searches: list[WebSearchItem] = Field(description="Lista wyszukiwań w sieci do wykonania, żeby najlepiej odpowiedzieć na zapytanie.")  # lista obiektów WebSearchItem

In [ ]:
# Podgląd schematu JSON wygenerowanego automatycznie przez Pydantic z klasy WebSearchPlan - identyczne dla OpenAI i Anthropic, bo to czysty Pydantic, bez udziału żadnego SDK dostawcy.

WebSearchPlan.model_json_schema()  # schemat JSON odpowiadający klasie WebSearchPlan

In [ ]:
# Patrz uwaga wyżej o koszcie WebSearchTool - Planner Agent sam nie wyszukuje, tylko planuje wyszukiwania dla Search Agenta.
# INSTRUCTIONS to f-string z HOW_MANY_SEARCHES (zdefiniowanym w komórce ze stałymi) - przetłumaczony na polski, bo to prompt systemowy.
# planner_agent to znów sam string instrukcji - output_format=WebSearchPlan zostanie przekazany do run() dopiero przy wywołaniu, nie tutaj (Agent(..., output_type=...) z oryginału łączył oba w jednym obiekcie).
# To ostatnia komórka w sekcji Agent 2 przed demo - wywołanie run() z tym agentem jest w kolejnej komórce.

INSTRUCTIONS = f"""
Jesteś asystentem badawczym. Na podstawie zapytania użytkownika wymyśl zestaw wyszukiwań w sieci,
które najlepiej odpowiedzą na to zapytanie. Wypisz {HOW_MANY_SEARCHES} terminów do wyszukania.
"""  # prompt systemowy Planner Agenta - UWAGA: mimo że po polsku, może skłonić Claude do generowania zapytań wyszukiwania po polsku (patrz notatka w komórce 0)

planner_agent = INSTRUCTIONS  # "Planner Agent" - string instrukcji

In [ ]:
# Demo Planner Agenta - ten sam task co w demo Search Agenta wyżej (celowo po angielsku), tym razem z output_format=WebSearchPlan.
# run() zwraca od razu sparsowany obiekt WebSearchPlan (odpowiednik result.final_output przy output_type=WebSearchPlan) - bez potrzeby rozpakowywania result.final_output ręcznie.
# plan.searches to lista obiektów WebSearchItem, każdy z polami query i reason - dokładnie ta struktura, którą run_searches() użyje niżej do wygenerowania rzeczywistych zapytań.
# W tej komórce, w przeciwieństwie do demo Search Agenta, nie ma żadnego wywołania narzędzia - Planner Agent tylko generuje strukturę danych.

plan = await run(planner_agent, task, output_format=WebSearchPlan)  # zwraca gotowy obiekt WebSearchPlan
plan  # podgląd sparsowanego planu wyszukiwań

## Agent 3: Writer Agent

In [ ]:
# Instrukcje Writer Agenta - przetłumaczone, bo to prompt systemowy.
# ReportData to kolejny schemat Pydantic (structured output), analogicznie do WebSearchPlan wyżej - Field(description=...) po polsku.
# writer_agent to znowu sam string instrukcji - output_format=ReportData trafi do run() dopiero przy wywołaniu w write_report() niżej.
# Writer Agent, tak jak Planner, nie dostaje żadnych narzędzi - jego jedynym zadaniem jest wygenerowanie ustrukturyzowanego raportu na podstawie dostarczonych wyników wyszukiwania.

INSTRUCTIONS = """
Jesteś starszym badaczem/badaczką, którego zadaniem jest napisanie spójnego raportu na podstawie zapytania badawczego.
Otrzymasz oryginalne zapytanie oraz wyniki researchu.
Wygeneruj obszerny raport na podstawie researchu i zapytania.
Finalny wynik powinien być w formacie markdown, obszerny i szczegółowy. Dąż do 5-10 stron treści, co najmniej 1000 słów.
"""  # prompt systemowy Writer Agenta


class ReportData(BaseModel):  # struktura raportu zwracana przez Writer Agenta
    short_summary: str = Field(description="Krótkie podsumowanie ustaleń w 2-3 zdaniach.")  # pole tekstowe - krótkie podsumowanie
    markdown_report: str = Field(description="Finalny raport.")  # pole tekstowe - pełny raport w markdown
    follow_up_questions: list[str] = Field(description="Sugerowane tematy do dalszego zbadania.")  # lista sugestii dalszych tematów


writer_agent = INSTRUCTIONS  # "Writer Agent" - string instrukcji

## Agent 4: Email agent

In [ ]:
# Odpowiednik dekoratora @function_tool z OpenAI Agents SDK - schemat piszemy ręcznie, bo Anthropic nie generuje go automatycznie z docstringa.
# Opis narzędzia (send_email_tool_json["description"]) jest DOSŁOWNYM tłumaczeniem oryginału - w oryginale ten opis też mówi o "sales prospects", mimo że to notatnik o Deep Research, nie o sprzedaży (najwyraźniej skopiowany z wcześniejszego labu bez adaptacji) - zachowujemy tę samą niespójność, zamiast ją po cichu poprawiać.
# Funkcja send_email_tool wywołuje send_email() albo push() (zaimportowane z messenger.py) w zależności od USE_EMAIL - identyczna logika jak w oryginale, tylko treść push jest teraz po polsku (to tekst, który faktycznie przeczytasz na telefonie).
# Nazwa funkcji musi się zgadzać z kluczem "name" w schemacie, bo nasza pętla run()/handle_tool_calls() szuka narzędzia po nazwie przez globals().get(block.name).

send_email_tool_json = {
    "name": "send_email_tool",  # nazwa narzędzia - musi się zgadzać z nazwą funkcji Pythona
    "description": "Wyślij e-mail o podanym temacie i treści do wszystkich potencjalnych klientów sprzedażowych",  # opis czytany przez Claude - dosłowne tłumaczenie oryginału, patrz komentarz wyżej
    "input_schema": {  # Anthropic używa klucza input_schema, nie parameters jak OpenAI, i nie generuje go automatycznie z docstringa
        "type": "object",
        "properties": {
            "subject": {"type": "string", "description": "Temat e-maila"},
            "text_body": {"type": "string", "description": "Treść e-maila jako czysty tekst"},
            "html_body": {"type": "string", "description": "Treść e-maila w formacie HTML"},
        },
        "required": ["subject", "text_body", "html_body"],
        "additionalProperties": False,
    },
}


def send_email_tool(subject: str, text_body: str, html_body: str) -> str:  # wersja funkcji send_email_tool jako narzędzie Claude
    if USE_EMAIL:  # jeśli e-mail jest skonfigurowany (patrz stała USE_EMAIL)
        send_email(subject, text_body, html_body)  # wyślij przez SMTP (funkcja z messenger.py)
    else:
        push(f"Temat: {subject}\n\n{text_body}")  # w przeciwnym razie wyślij push z tematem i treścią tekstową
    return "E-mail wysłany pomyślnie"  # ten string trafi do Claude jako tool_result

In [ ]:
# Podgląd schematu JSON narzędzia, który Claude czyta przy decyzji, jak wypełnić argumenty.
# W OpenAI Agents SDK ten schemat (params_json_schema) jest generowany automatycznie z sygnatury funkcji i docstringa.
# W naszej wersji nie ma żadnej magii dekoratora - schemat to zwykły klucz "input_schema" w słowniku send_email_tool_json, wpisany ręcznie w komórce wyżej.
# To jedyna linia w tej komórce, więc różni się od oryginalnego send_email_tool.params_json_schema tylko sposobem dostępu do tych samych informacji.

send_email_tool_json["input_schema"]  # schemat argumentów narzędzia (odpowiednik send_email_tool.params_json_schema)

In [ ]:
# Instrukcje Email Agenta - przetłumaczone, bo to prompt systemowy.
# email_agent to znów sam string instrukcji; email_tools to lista narzędzi dostępnych temu agentowi (tylko jedno: send_email_tool_json).
# W przeciwieństwie do Search Agenta, Email Agent NIE dostaje wymuszonego tool_choice - tak jak w oryginale, polega wyłącznie na instrukcji w prompcie ("Use your tool to send an email").
# To ostatni z czterech agentów zdefiniowanych w tym notatniku - kolejne komórki orkiestrują ich współpracę przez kod.

INSTRUCTIONS = """
Otrzymujesz szczegółowy raport. Użyj swojego narzędzia, żeby wysłać e-mail, przekształcając raport w
czysty, dobrze zaprezentowany e-mail HTML z odpowiednim tematem.
"""  # prompt systemowy Email Agenta

email_agent = INSTRUCTIONS  # "Email Agent" - string instrukcji
email_tools = [send_email_tool_json]  # lista narzędzi dostępnych temu agentowi

## Czas na orkiestrację przez kod

Kolejne 2 funkcje zaplanują i wykonają wyszukiwanie, korzystając z Agentów, przez wywołania `run()` (odpowiednik `Runner.run()`).

In [ ]:
# run_searches() planuje wyszukiwania przez Planner Agenta, potem wykonuje je RÓWNOLEGLE przez Search Agenta (asyncio.gather) - ten sam wzorzec równoległości co w 2_lab2.pl.ipynb.
# f"Zapytanie: {query}" to szablon promptu wysyłany do Plannera - w przeciwieństwie do demo w komórce wyżej, gdzie task szedł bez żadnego prefiksu (dokładnie jak w oryginale).
# search() woła Search Agenta z tools=search_tools i tool_choice=search_tool_choice zdefiniowanymi w komórce z instrukcjami Search Agenta - wymusza faktyczne wyszukiwanie, nie tylko opis, co by wyszukał.
# item: WebSearchItem to obiekt Pydantic zwrócony przez Planner Agenta (patrz WebSearchPlan.searches) - item.query i item.reason to jego pola.

async def run_searches(query: str):  # planuje i wykonuje wszystkie wyszukiwania dla danego zapytania
    print("Planowanie wyszukiwań...")  # komunikat postępu
    plan = await run(planner_agent, f"Zapytanie: {query}", output_format=WebSearchPlan)  # zaplanuj wyszukiwania - zwraca gotowy obiekt WebSearchPlan
    searches = plan.searches  # lista obiektów WebSearchItem
    print(f"Zostanie wykonanych {len(searches)} wyszukiwań")  # komunikat postępu z liczbą wyszukiwań
    tasks = [search(item) for item in searches]  # przygotuj listę korutyn - po jednej na wyszukiwanie
    results = await asyncio.gather(*tasks)  # wykonaj wszystkie wyszukiwania równolegle
    print("Zakończono wyszukiwanie")  # komunikat postępu
    return results  # lista podsumowań tekstowych - po jednym na wyszukiwanie


async def search(item: WebSearchItem):  # wykonuje jedno wyszukiwanie przez Search Agenta
    input_message = f"Termin wyszukiwania: {item.query}\nPowód wyszukiwania: {item.reason}"  # szablon promptu z polami obiektu WebSearchItem
    result = await run(search_agent, input_message, tools=search_tools, tool_choice=search_tool_choice)  # wymuszone wyszukiwanie w sieci, patrz komórka z definicją search_agent
    return result  # podsumowanie tekstowe wyniku wyszukiwania

Kolejne 2 funkcje napiszą raport i wyślą go e-mailem

In [ ]:
# write_report() woła Writer Agenta ze structured output (output_format=ReportData) - run() zwraca od razu gotowy obiekt ReportData, bez result.final_output.
# send_report_email() woła Email Agenta BEZ output_format - run() zwraca zwykły tekst (finalna odpowiedź Email Agenta po wysłaniu e-maila), tools=email_tools daje mu dostęp do send_email_tool.
# Oba szablony promptów (input_message) są przetłumaczone na polski - to tekst wysyłany do modelu.
# Te dwie funkcje razem z run_searches()/search() z komórki wyżej to komplet czterech funkcji orkiestrujących - po jednej "ścieżce wywołania" na każdego z czterech agentów.

async def write_report(query: str, search_results: list[str]):  # pisze raport na podstawie zapytania i wyników wyszukiwań
    print("Praca nad raportem...")  # komunikat postępu
    input_message = f"Oryginalne zapytanie: {query}\nPodsumowane wyniki wyszukiwania: {search_results}"  # szablon promptu dla Writer Agenta
    report = await run(writer_agent, input_message, output_format=ReportData)  # zwraca gotowy obiekt ReportData
    print("Zakończono pisanie raportu")  # komunikat postępu
    return report  # obiekt ReportData (short_summary, markdown_report, follow_up_questions)


async def send_report_email(report: ReportData):  # wysyła raport e-mailem przez Email Agenta
    print("Pisanie e-maila...")  # komunikat postępu
    result = await run(email_agent, report.markdown_report, tools=email_tools)  # Email Agent sam decyduje, kiedy wywołać send_email_tool
    print("E-mail wysłany")  # komunikat postępu
    return result  # finalna odpowiedź tekstowa Email Agenta

### Czas na pokaz!

In [ ]:
# Pełny przepływ end-to-end: planowanie + wyszukiwanie -> napisanie raportu -> wysyłka e-mailem, opakowane w trace() - jedyne miejsce w tym notatniku, gdzie trace() faktycznie coś opakowuje.
# query zostaje po angielsku, tak jak task wyżej - patrz notatka architektoniczna w komórce 0 (jakość wyszukiwania dla tego konkretnego tematu).
# To wywołanie realnie kosztuje: kilka wyszukiwań w sieci przez web_search i (jeśli USE_EMAIL=True oraz dane SMTP są ustawione) wysyła prawdziwy e-mail.
# Kolejność await-ów w tym bloku jest sekwencyjna, nie równoległa (w przeciwieństwie do wnętrza run_searches()) - raport i e-mail muszą poczekać, aż wszystkie wyszukiwania się skończą.

query = "Most popular AI Agent frameworks in 2026"  # przykładowe zapytanie - celowo po angielsku (patrz notatka w komórce 0)

with trace("Trace badawczy"):  # jeden znacznik czasu obejmujący cały przepływ
    print("Rozpoczynam badanie...")  # komunikat startu
    search_results = await run_searches(query)  # zaplanuj i wykonaj wszystkie wyszukiwania
    report = await write_report(query, search_results)  # napisz raport na podstawie wyników
    await send_report_email(report)  # wyślij raport e-mailem
    print("Hurra!")  # komunikat końca

### Sprawdź trace

Tym razem `trace()` faktycznie coś opakował (patrz komórka wyżej) - ale tak jak wcześniej, to tylko dwie linie w konsoli (start/koniec + czas trwania), nie prawdziwa platforma trace jak `platform.openai.com/traces` w oryginale.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Gratulacje z okazji postępów i mała prośba</h2>
            <span style="color:#00cc00;">Dotarłeś do ważnego momentu tego kursu - stworzyłeś wartościowego Agenta, korzystając z jednego z najnowszych frameworków agentowych. Podniosłeś swoje kompetencje i odblokowałeś nowe możliwości komercyjne. Poświęć chwilę, żeby to uczcić!<br/><br/>Coś, o co powinienem Cię poprosić - mój redaktor by mnie strofował, gdybym o tym nie wspomniał. Jeśli możesz ocenić kurs na Udemy, byłbym naprawdę wdzięczny: to najważniejszy sposób, w jaki Udemy decyduje, czy pokazywać kurs innym, i robi to ogromną różnicę.<br/><br/>I kolejne przypomnienie, żeby <a href="https://www.linkedin.com/in/eddonner/">połączyć się ze mną na LinkedIn</a>, jeśli chcesz! Jeśli chciałbyś napisać o swoich postępach w kursie, oznacz mnie, a dołożę się, żeby zwiększyć Twój zasięg.
            </span>
        </td>
    </tr>
</table>